# BGE-M3 ONNX CPU FP32 Benchmark Results
## Active Analysis Notebook

**Data**: 24 runs — full factorial of `batch_size` ∈ {1, 8, 16, 32} ×
`max_length` ∈ {64, 128} × `language` ∈ {en, th, mixed}.  1 run per cell.

**Runtime**: Run `uv sync` in this directory, then `uv run jupyter notebook`.

**Path**: This notebook lives in `notebooks/`, results are in `../results/`.

**Machine metadata** (`machine_metadata` field) is present — cells handle it
gracefully and surface CPU model, core count, and RAM automatically.

In [ ]:
# ── Install required packages via uv ─────────────────────────────────────────
!uv pip install pandas matplotlib numpy jupyter seaborn ipywidgets -q


## Step 1 — Load Data + Language Filter

Load `onnx_cpu_fp32.jsonl` and expand nested `machine_metadata`.
Use the **language selector** below to filter runs.  All 24 runs are loaded
initially; charts below update to show only the selected language.

| Language | Dataset description |
|----------|--------------------|
| `en`    | English Wikipedia |
| `th`    | Thai Wikipedia |
| `mixed` | 50/50 English + Thai |
| `all`   | All three combined |

In [ ]:
# ── Setup + Language Filter ────────────────────────────────────────────────
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

DATA_PATH = Path("../results/onnx_cpu_fp32.jsonl")

records_raw = []
with open(DATA_PATH) as f:
    for line in f:
        records_raw.append(json.loads(line))

def expand_record(r):
    out = {k: v for k, v in r.items() if k != "machine_metadata"}
    mm = r.get("machine_metadata", {}) or {}
    out["cpu_model"]          = mm.get("cpu", {}).get("model_name", "unknown")
    out["cpu_physical_cores"] = mm.get("cpu", {}).get("physical_cores", "?")
    out["cpu_logical_cores"] = mm.get("cpu", {}).get("logical_cores", "?")
    out["ram_gb"]            = round(mm.get("memory", {}).get("total_bytes", 0) / 1e9, 1)
    out["onnxruntime_version"] = mm.get("runtime", {}).get("onnxruntime_version", "?")
    out["python_version"]      = mm.get("runtime", {}).get("python_version", "?")
    return out

records_all = pd.DataFrame(expand_record(r) for r in records_raw)
records_all["run"] = records_all.index + 1

# ── Bottleneck classification ───────────────────────────────────────────────
tok_ms = records_all["tokenize_latency_ms_avg"]
emb_ms = records_all["embedding_latency_ms_avg"]
records_all["emb_ratio"] = emb_ms / (tok_ms + emb_ms)

def classify(emb_ratio):
    if emb_ratio > 0.90: return "model-bound"
    elif emb_ratio < 0.50: return "tokenize-heavy"
    else: return "balanced"

records_all["bottleneck_type"] = records_all["emb_ratio"].apply(classify)

def get_filtered(language):
    if language == "all":
        return records_all
    return records_all[records_all["language"] == language].copy()

lang_dropdown = widgets.Dropdown(
    options=[("English", "en"), ("Thai", "th"), ("Mixed", "mixed"), ("All languages", "all")],
    value="all",
    description="Language:",
    style={"description_width": "80px"},
    layout={"width": "260px"},
)

def on_lang_change(change):
    global records
    lang = change["new"]
    records = get_filtered(lang)
    clear_output(wait=True)
    display(lang_dropdown)
    print(f"\nShowing {len(records)} runs for language='{lang}'  "
          f"(batch_sizes={sorted(records['batch_size'].unique())}, "
          f"max_lengths={sorted(records['max_length'].unique())})")
    display(records[["run","language","batch_size","max_length",
                    "embedding_tokens_per_sec","end_to_end_tokens_per_sec","bottleneck_type"]])

lang_dropdown.observe(on_lang_change, names="value")
display(lang_dropdown)

records = get_filtered("all")
print(f"\nLoaded {len(records)} runs across batch_size × max_length × language")
print(f"batch_sizes : {sorted(records['batch_size'].unique())}")
print(f"max_lengths : {sorted(records['max_length'].unique())}")
print(f"languages   : {sorted(records['language'].unique())}")
display(records[["run","language","batch_size","max_length",
                "embedding_tokens_per_sec","end_to_end_tokens_per_sec","bottleneck_type"]])


## Step 2 — Summary Statistics (Aggregate Across All Loaded Runs)

Shows mean / median / std / CV% / min / max **per metric** across all runs
loaded for the selected language (use the selector above to filter).

Key signals:
- **embedding tok/s** — primary ONNX bottleneck metric
- **end-to-end tok/s** — real-world throughput including tokenizer
- **emb_ratio** — fraction of total time in embedding stage
- **bottleneck_type** counts — most runs are model-bound (embedding dominates)

In [ ]:
# ── Summary statistics table ─────────────────────────────────────────────
metrics = [
    ("embedding_tokens_per_sec",  "Embedding tok/s"),
    ("end_to_end_tokens_per_sec","End-to-end tok/s"),
    ("tokenize_tokens_per_sec", "Tokenization tok/s"),
    ("embedding_latency_ms_avg",  "Embedding latency ms"),
    ("end_to_end_latency_ms_avg","End-to-end latency ms"),
    ("emb_ratio",               "Embedding ratio"),
]

rows = []
for metric, label in metrics:
    v = records[metric]
    rows.append([
        label,
        f"{v.mean():.2f}",
        f"{v.median():.2f}",
        f"{v.std():.2f}",
        f"{v.std()/v.mean()*100:.1f}%",
        f"{v.min():.2f}",
        f"{v.max():.2f}",
    ])

stats_df = pd.DataFrame(rows, columns=["Metric", "Mean", "Median", "Std", "CV%", "Min", "Max"])

styled = (
    stats_df.style
    .format(precision=2)
    .hide(axis="index")
    .set_table_attributes('style="border-collapse:collapse;font-size:13px;margin-top:8px"')
    .set_caption(f"Summary Statistics — {len(records)} Runs")
)
display(styled)

print(f"\nBottleneck breakdown:")
display(records["bottleneck_type"].value_counts())


## Step 3 — Run-level Results + Bottleneck Classification

Full per-run table for the selected language, **sorted by language → max_length
→ batch_size** and including the `language` column (so runs with the same
`bs`/`max_length` across languages are distinguishable).

Throughput columns carry **inline data bars** for at-a-glance comparison.
The `bottleneck_type` cell is color-coded:

| Background | Type | emb_ratio |
|------------|------|-----------|
| 🔵 light blue | model-bound | > 90% |
| 🟠 light orange | tokenize-heavy | < 50% |
| 🟢 light green | balanced | 50–90% |

(On this CPU-FP32 data every run is **model-bound** — embedding is ~99.7% of
compute time — so the whole column reads blue.)

In [ ]:
# ── Run-level results table ────────────────────────────────────────────────
table_cols = [
    "run", "language", "batch_size", "max_length",
    "embedding_tokens_per_sec", "end_to_end_tokens_per_sec",
    "emb_ratio", "bottleneck_type",
]
table_cols = [c for c in table_cols if c in records.columns]

# Sort by language → max_length → batch_size so runs are scannable and grouped
run_table = (records[table_cols]
             .sort_values([c for c in ["language", "max_length", "batch_size"] if c in table_cols])
             .reset_index(drop=True))

bottleneck_colors = {
    "model-bound":    "#DBEAFE",
    "tokenize-heavy": "#FFEDD5",
    "balanced":       "#DCFCE7",
}

def color_bottleneck(col):
    # dark, bold text on the tinted background for strong contrast
    return [f"background-color: {bottleneck_colors.get(v, '')}; color:#0f172a; font-weight:700"
            for v in col]

num_fmt = {
    "embedding_tokens_per_sec":  "{:,.0f}",
    "end_to_end_tokens_per_sec": "{:,.0f}",
    "emb_ratio":                 "{:.1%}",
}
num_fmt = {k: v for k, v in num_fmt.items() if k in run_table.columns}
bar_cols = [c for c in ["embedding_tokens_per_sec", "end_to_end_tokens_per_sec"]
            if c in run_table.columns]

# Every element gets an explicit light background + dark text, so the table reads
# as a self-contained light "card" regardless of the Jupyter theme (light OR dark).
styled_table = (
    run_table.style
    .format(num_fmt)
    .set_properties(**{"background-color": "#ffffff", "color": "#111827"})  # base cells
    .apply(color_bottleneck, subset=["bottleneck_type"])                   # tinted classification
    .bar(subset=bar_cols, color="#93c5fd", vmin=0)                         # data bars (over white base)
    .set_table_styles([
        {"selector": "",
         "props": [("background-color", "#ffffff"), ("color", "#111827")]},
        {"selector": "th",
         "props": [("color", "#0f172a"), ("background-color", "#e2e8f0"),
                   ("font-weight", "700"), ("border", "1px solid #cbd5e1"),
                   ("padding", "4px 8px")]},
        {"selector": "td",
         "props": [("color", "#111827"), ("border", "1px solid #e2e8f0"),
                   ("padding", "3px 8px")]},
        {"selector": "caption",
         "props": [("color", "#0f172a"), ("background-color", "#ffffff"),
                   ("font-weight", "700"), ("font-size", "13px"),
                   ("caption-side", "top"), ("padding", "6px 8px")]},
    ])
    .hide(axis="index")
    .set_caption(f"Run-level results — {len(run_table)} runs (sorted by language → max_length → batch_size)")
    .set_table_attributes('style="border-collapse:collapse;font-size:12px;margin-top:8px;'
                          'background-color:#ffffff;color:#111827"')
)
display(styled_table)


## Step 4 — Per-run Latency Breakdown (Faceted by max_length)

One **row per run** (`bs · language`), with three grouped bars per row —
**Tokenize** (amber) / **Embedding** (blue) / **End-to-end** (green). Each row is
labeled with **both units**: end-to-end `ms` **and** its throughput in
`inputs/sec` (`end_to_end_items_per_sec`) — latency for "how slow is one batch",
inputs/sec for "how many can I process".

Encoding:
- **Bar color** = pipeline stage
- **Row background tint** = language (en / th / mixed)
- **Facet** = `max_length`; rows ordered by `batch_size` then language

The x-axis is **log-scaled** — without it, tokenization (~0.5–20 ms) is
invisible next to embedding (~130 ms – 12 s). End-to-end can dip *below*
embedding because the tokenizer and ONNX model overlap in the pipeline.

In [ ]:
# ── Latency breakdown: one row per (batch_size, language), faceted by max_length ──
import numpy as np
from matplotlib.patches import Patch

STAGE_DEFS = [
    ("tokenize_latency_ms_p95",   "Tokenize",   "#f59e0b"),
    ("embedding_latency_ms_p95",  "Embedding",  "#3b82f6"),
    ("end_to_end_latency_ms_p95", "End-to-end", "#10b981"),
]
LANG_ORDER = ["en", "th", "mixed"]
LANG_TINT  = {"en": "#e8f0fe", "th": "#fde8e8", "mixed": "#f3e8fd"}  # row background = language

max_lengths   = sorted(records["max_length"].unique())
langs_present = [l for l in LANG_ORDER if l in records["language"].unique()]
multi_lang    = len(langs_present) > 1
lang_rank     = {l: i for i, l in enumerate(LANG_ORDER)}

fig, axes = plt.subplots(1, len(max_lengths), figsize=(8.5 * len(max_lengths), 9),
                         sharex=True)
axes = np.atleast_1d(axes)

bar_h, step = 0.24, 0.26   # 3 bars span ±0.37 inside each ±0.5 row band → clear gap between runs

for ax, ml in zip(axes, max_lengths):
    sub = records[records["max_length"] == ml].copy()
    sub["_lr"] = sub["language"].map(lang_rank)
    sub = sub.sort_values(["batch_size", "_lr"]).reset_index(drop=True)
    n = len(sub)
    y = np.arange(n)

    # Row backgrounds: tint by language (multi) or zebra (single language)
    for i, row in sub.iterrows():
        band = (LANG_TINT.get(row["language"], "#f0f0f5") if multi_lang
                else ("#ffffff" if i % 2 == 0 else "#ececf2"))
        ax.axhspan(i - 0.5, i + 0.5, color=band, zorder=0)
    ax.set_ylim(-0.5, n - 0.5)
    ax.set_axisbelow(True)

    # Grouped stage bars
    for j, (col, name, color) in enumerate(STAGE_DEFS):
        ax.barh(y + (j - 1) * step, sub[col], height=bar_h, color=color,
                label=name, alpha=0.95, edgecolor="white", linewidth=0.5, zorder=3)

    # Annotate end-to-end latency AND its throughput (inputs/sec) — easier to digest
    for i, (e2e, ips) in enumerate(zip(sub["end_to_end_latency_ms_p95"],
                                       sub["end_to_end_items_per_sec"])):
        ax.text(e2e * 1.18, i + step, f"{e2e:,.0f} ms · {ips:,.1f} inp/s", va="center",
                fontsize=7, color="#065f46", zorder=4)

    ylabels = ([f"bs={int(b)} · {l}" for b, l in zip(sub["batch_size"], sub["language"])]
               if multi_lang else [f"bs={int(b)}" for b in sub["batch_size"]])
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlim(0.3, 80000)
    ax.set_xlabel("p95 latency ms  (log scale)")
    ax.set_title(f"max_length = {ml}   ({n} runs)", fontsize=12, fontweight="bold", pad=8)
    ax.grid(axis="x", which="both", alpha=0.35)
    ax.grid(axis="y", visible=False)

# Stage legend (left facet) + language-tint legend (right facet, when multi-language)
axes[0].legend(handles=[Patch(facecolor=c, label=n) for _, n, c in STAGE_DEFS],
               loc="lower right", fontsize=9, title="Pipeline stage", framealpha=0.95)
if multi_lang:
    axes[-1].legend(handles=[Patch(facecolor=LANG_TINT[l], edgecolor="#bbbbbb", label=l)
                             for l in langs_present],
                    loc="lower right", fontsize=9, title="Language (row tint)", framealpha=0.95)

lang_label = "all" if multi_lang else langs_present[0]
fig.suptitle(f"BGE-M3 ONNX CPU FP32 — p95 Latency Breakdown   |   language: {lang_label}   "
             f"|   labels show ms + end-to-end inputs/sec",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## Step 4b — Per-Input Cost (ms per input **and** inputs/sec)

Step 4 showed per-*batch* latency; this normalizes cost to a **single input item**,
shown in **both units** so it's easy to digest:

- bottom x-axis = **ms per input** — `1000 / *_items_per_sec` (cost)
- top x-axis = **inputs / sec** — `*_items_per_sec` (throughput); the same bars,
  just read as a rate (`1000 ÷ ms-per-input`)

Two bars per run — **Tokenize/input** (amber) vs **Embedding/input** (blue) — on a
**log x-axis** (they differ ~100×). Each y-label carries **tokens per input**
(`avg_tokens_per_item`), and row tint encodes language. The embedding bar is
annotated with both `ms` and `/s`.

Key insight: unlike per-batch latency, per-input embedding cost **drops as batch
size grows** (inputs/sec **rises**) — batching amortizes the model's fixed
per-call overhead across more inputs.

In [ ]:
# ── Per-input cost: tokens + tokenize/embedding ms PER INPUT, faceted by max_length ──
import numpy as np
from matplotlib.patches import Patch

per_input = records.copy()
per_input["tokenize_ms_per_input"]  = 1000.0 / per_input["tokenize_items_per_sec"]
per_input["embedding_ms_per_input"] = 1000.0 / per_input["embedding_items_per_sec"]

STAGE_DEFS = [
    ("tokenize_ms_per_input",  "Tokenize / input",  "#f59e0b"),
    ("embedding_ms_per_input", "Embedding / input", "#3b82f6"),
]
LANG_ORDER = ["en", "th", "mixed"]
LANG_TINT  = {"en": "#e8f0fe", "th": "#fde8e8", "mixed": "#f3e8fd"}  # row background = language

max_lengths   = sorted(per_input["max_length"].unique())
langs_present = [l for l in LANG_ORDER if l in per_input["language"].unique()]
multi_lang    = len(langs_present) > 1
lang_rank     = {l: i for i, l in enumerate(LANG_ORDER)}

# ms-per-input ↔ inputs-per-sec is a self-inverse reciprocal (1000 / x), so the same
# function serves both directions of the secondary "inputs/sec" axis.
def ms_ips(x):
    x = np.asarray(x, dtype=float)
    return np.divide(1000.0, x, out=np.full_like(x, np.nan), where=x > 0)

fig, axes = plt.subplots(1, len(max_lengths), figsize=(8.5 * len(max_lengths), 9),
                         sharex=True)
axes = np.atleast_1d(axes)

bar_h, step = 0.30, 0.34   # 2 bars span ±0.32 inside each ±0.5 row band → clear gap between runs

for ax, ml in zip(axes, max_lengths):
    sub = per_input[per_input["max_length"] == ml].copy()
    sub["_lr"] = sub["language"].map(lang_rank)
    sub = sub.sort_values(["batch_size", "_lr"]).reset_index(drop=True)
    n = len(sub)
    y = np.arange(n)

    # Row backgrounds: tint by language (multi) or zebra (single language)
    for i, row in sub.iterrows():
        band = (LANG_TINT.get(row["language"], "#f0f0f5") if multi_lang
                else ("#ffffff" if i % 2 == 0 else "#ececf2"))
        ax.axhspan(i - 0.5, i + 0.5, color=band, zorder=0)
    ax.set_ylim(-0.5, n - 0.5)
    ax.set_axisbelow(True)

    # Two grouped bars: tokenize vs embedding cost per single input
    for j, (col, name, color) in enumerate(STAGE_DEFS):
        ax.barh(y + (j - 0.5) * step, sub[col], height=bar_h, color=color,
                label=name, alpha=0.95, edgecolor="white", linewidth=0.5, zorder=3)

    # Annotate the dominant cost in BOTH units: ms/input and inputs/sec
    for i, v in enumerate(sub["embedding_ms_per_input"]):
        ax.text(v * 1.15, i + 0.5 * step, f"{v:,.0f} ms · {1000.0 / v:,.1f}/s", va="center",
                fontsize=7, color="#1e3a8a", zorder=4)

    # y-label carries tokens-per-input so all three quantities are visible
    if multi_lang:
        ylabels = [f"bs={int(b)} · {l} · {int(t)} tok"
                   for b, l, t in zip(sub["batch_size"], sub["language"], sub["avg_tokens_per_item"])]
    else:
        ylabels = [f"bs={int(b)} · {int(t)} tok"
                   for b, t in zip(sub["batch_size"], sub["avg_tokens_per_item"])]
    ax.set_yticks(y)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xscale("log")
    ax.set_xlim(0.05, 2500)
    ax.set_xlabel("milliseconds per input  (log scale)")
    ax.set_title(f"max_length = {ml}   ({n} runs)", fontsize=12, fontweight="bold", pad=8)
    ax.grid(axis="x", which="both", alpha=0.35)
    ax.grid(axis="y", visible=False)

    # Secondary top axis: the SAME bars read as inputs/sec (throughput) — easier to digest
    secax = ax.secondary_xaxis("top", functions=(ms_ips, ms_ips))
    secax.set_xlabel("inputs / sec   (= 1000 ÷ ms-per-input)", fontsize=10)

# Stage legend (left facet) + language-tint legend (right facet, when multi-language)
axes[0].legend(handles=[Patch(facecolor=c, label=n) for _, n, c in STAGE_DEFS],
               loc="lower right", fontsize=9, title="Per-input stage", framealpha=0.95)
if multi_lang:
    axes[-1].legend(handles=[Patch(facecolor=LANG_TINT[l], edgecolor="#bbbbbb", label=l)
                             for l in langs_present],
                    loc="lower right", fontsize=9, title="Language (row tint)", framealpha=0.95)

lang_label = "all" if multi_lang else langs_present[0]
fig.suptitle(f"BGE-M3 ONNX CPU FP32 — Per-Input Cost: ms-per-input (bottom) + inputs/sec (top)   "
             f"|   language: {lang_label}",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


## Step 5 — Embedding vs End-to-end Throughput

Each point = one run.  **Color** = `max_length`, **Size** = `batch_size`.

Points above the diagonal = embedding is the bottleneck (e2e ≈ emb).
Points far below diagonal = significant tokenization overhead.

With 3 languages × 4 batch_sizes × 2 max_lengths = 24 runs, this chart
shows the full pareto frontier of throughput vs max_length tradeoffs.

In [ ]:
# ── Scatter: embedding vs end-to-end throughput ───────────────────────────
fig, ax = plt.subplots(figsize=(18, 10))

ml_colors  = {64: "#3b82f6", 128: "#f97316"}
ml_markers = {64: "o",        128: "s"}
ml_labels = {64: "max_length=64", 128: "max_length=128"}
lang_styles = {"en": {}, "th": {"hatch": "///"}, "mixed": {"hatch": "\\\\\\"}}

for ml in sorted(records["max_length"].unique()):
    sub = records[records["max_length"] == ml]
    sizes = (sub["batch_size"] / records["batch_size"].max() * 300 + 80)

    ax.scatter(
        sub["embedding_tokens_per_sec"],
        sub["end_to_end_tokens_per_sec"],
        s=sizes,
        color=ml_colors.get(ml, "gray"),
        alpha=0.82,
        label=ml_labels.get(ml, str(ml)),
        edgecolors="white",
        linewidths=2,
        zorder=5,
        marker=ml_markers.get(ml, "o"),
    )
    for _, row in sub.iterrows():
        ax.annotate(
            f"bs={int(row['batch_size'])}",
            (row["embedding_tokens_per_sec"], row["end_to_end_tokens_per_sec"]),
            textcoords="offset points", xytext=(7, 7),
            fontsize=9, color="#1f2937", fontweight="bold",
        )

emb_min, emb_max = records["embedding_tokens_per_sec"].min(), records["embedding_tokens_per_sec"].max()
e2e_min, e2e_max = records["end_to_end_tokens_per_sec"].min(), records["end_to_end_tokens_per_sec"].max()

ax.axhline((e2e_min + e2e_max) / 2, color="gray", linestyle="-", alpha=0.12, linewidth=1)
ax.axvline((emb_min + emb_max) / 2, color="gray", linestyle="-", alpha=0.12, linewidth=1)

diag_min, diag_max = max(emb_min, e2e_min) * 0.95, min(emb_max, e2e_max) * 1.05
ax.plot([diag_min, diag_max], [diag_min, diag_max], "--",
        color="gray", alpha=0.45, linewidth=1.5,
        label="emb = e2e (no tokenization overhead)")

ax.set_xlim(emb_min * 0.95, emb_max * 1.05)
ax.set_ylim(e2e_min * 0.95, e2e_max * 1.05)
ax.set_xlabel("Embedding tok/s  (higher = faster ONNX)", fontsize=12)
ax.set_ylabel("End-to-end tok/s  (higher = faster pipeline)", fontsize=12)
ax.set_title(
    "Embedding vs End-to-end Throughput\n"
    "(color = max_length | marker = max_length | size = batch_size | label = batch_size)",
    fontsize=13, pad=12
)
ax.legend(fontsize=10, title="Sequence length", title_fontsize=10,
         loc="lower right", framealpha=0.9)
ax.grid(alpha=0.20, linewidth=0.8)

# annotation for key range info
ax.text(0.98, 0.02,
        f"emb range: {emb_min:.0f}–{emb_max:.0f} tok/s\n"
        f"e2e range: {e2e_min:.0f}–{e2e_max:.0f} tok/s",
        transform=ax.transAxes, fontsize=9, va="bottom", ha="right",
        color="#6b7280",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.7))

plt.tight_layout()
plt.show()


## Step 6 — Batch Scaling: Throughput + Efficiency vs Batch Size

Both panels plot vs `batch_size` (evenly spaced — 1/8/16/32 aren't linear), one
line per `max_length`. Because each `(max_length, batch_size)` has one run per
language, lines show the **median across languages**, with a **min–max band** and
**faint per-run dots** so language spread is visible without zig-zagging.

**Top** — `embedding_tokens_per_sec`: compute vs memory-bound tradeoff.

**Bottom** — efficiency = `tok/s ÷ batch_size` on a **log y-axis**: normalizes for
batch size; the `bs=1` value dwarfs the rest, so log scale reveals the structure.

In [ ]:
# ── Batch scaling: throughput + efficiency (median across languages + spread band) ──
import numpy as np

ml_colors  = {64: "#3b82f6", 128: "#f97316"}
ml_markers = {64: "o",        128: "s"}
ml_labels  = {64: "max_length=64", 128: "max_length=128"}

batch_vals  = sorted(records["batch_size"].unique())
x_pos       = {b: i for i, b in enumerate(batch_vals)}   # even spacing — 1/8/16/32 aren't linear
max_lengths = sorted(records["max_length"].unique())

eff = records.copy()
eff["efficiency"] = eff["embedding_tokens_per_sec"] / eff["batch_size"]

# (column, y-label, title, log-y?, legend-loc)
panels = [
    ("embedding_tokens_per_sec", "Embedding tok/s",
     "Embedding Throughput vs Batch Size", False, "lower right"),
    ("efficiency", "Efficiency (tok/s ÷ batch_size)",
     "Batch Efficiency: tok/s ÷ batch_size   (log y — the bs=1 value dwarfs the rest)",
     True, "upper right"),
]

fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)

for ax, (col, ylabel, title, logy, loc) in zip(axes, panels):
    for ml in max_lengths:
        sub = eff[eff["max_length"] == ml]
        g   = sub.groupby("batch_size")[col]
        med, lo, hi = g.median(), g.min(), g.max()
        xs  = [x_pos[b] for b in med.index]
        color = ml_colors.get(ml, "gray")

        # min–max spread across languages
        ax.fill_between(xs, lo.values, hi.values, color=color, alpha=0.12, zorder=2)
        # faint individual runs (one dot per language)
        ax.scatter([x_pos[b] for b in sub["batch_size"]], sub[col],
                   color=color, alpha=0.35, s=26, edgecolors="none", zorder=3)
        # bold median curve
        ax.plot(xs, med.values, marker=ml_markers.get(ml, "o"), markersize=10,
                linewidth=2.8, color=color, label=ml_labels.get(ml, str(ml)), zorder=5)
        for xi, mv in zip(xs, med.values):
            ax.annotate(f"{mv:.0f}", (xi, mv), textcoords="offset points",
                        xytext=(0, 11), ha="center", fontsize=9, fontweight="bold",
                        color=color, zorder=6)

    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, pad=10)
    if logy:
        ax.set_yscale("log")
    ax.grid(alpha=0.30, linewidth=0.8, which="both")
    ax.legend(fontsize=10, title="Sequence length", title_fontsize=10,
              loc=loc, framealpha=0.9)

axes[-1].set_xticks(list(x_pos.values()))
axes[-1].set_xticklabels([f"bs={b}" for b in batch_vals])
axes[-1].set_xlabel("Batch Size", fontsize=12)
axes[-1].set_xlim(-0.35, len(batch_vals) - 0.65)

lang_label = "all" if records["language"].nunique() > 1 else records["language"].unique()[0]
fig.suptitle(f"BGE-M3 ONNX CPU FP32 — Batch Scaling   |   language: {lang_label}   "
             f"|   line = median across languages · band = min–max · dots = individual runs",
             fontsize=15, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


## Quick Reference — Adding More Benchmark Runs

| Variation | What to run | What the charts show |
|-----------|-------------|----------------------|
| `batch_size` sweep | `batch_size`={1,2,4,8,16,32} | Step 6: batch scaling curve |
| `max_length` sweep | `max_length`={64,128,256,512} | Steps 4–6: seq-length tradeoff |
| `language` sweep | `language`={en,th,mixed} | Steps 1–5: language-specific charts |